In [ ]:
!pip install adjustText


**Normalizing** **to glc mean**

In [ ]:
import pandas as pd

# File paths for different tissue flux data
file_paths = [
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Breast_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\BronchusLung_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Colon_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Kidney_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Liver_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Prostate_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Stomach_detailed_means.csv",
    r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\MEANS\Thyroid_detailed_means.csv",
]

# Output directory for normalized flux data
output_dir = r"D:\FLUX SAMPLING MEAN DATA-20250126T133616Z-001\FLUX SAMPLING MEAN DATA\MEAN FLUXES\NORMALIZED MEANS GLC"


glucose_rxn = "EX_glc(e)"
normalized_tissue_dict = {}

# Process each file
for file_path in file_paths:
    tissue_name = file_path.split("\\")[-1].replace("_detailed_means.csv", "")  # Extract tissue name
    df = pd.read_csv(file_path)

    if glucose_rxn in df['Reaction'].values:
        for column in df.columns:
            if column != 'Reaction':
                glucose_value = df.loc[df['Reaction'] == glucose_rxn, column].values[0]
                glucose_value = abs(glucose_value)

                if glucose_value != 0:
                    df[column] = df[column] / glucose_value

        # Save the normalized data for this tissue
        df.to_csv(f"{output_dir}/normalized_{tissue_name}.csv", index=False)
        normalized_tissue_dict[tissue_name] = df

# Merge all normalized data into a single CSV file
if normalized_tissue_dict:
    merged_df = pd.concat(normalized_tissue_dict.values(), ignore_index=True)
    merged_df.to_csv(f"{output_dir}/normalized_all_samples.csv", index=False)

print("Normalization complete. Files saved in:", output_dir)


**LINEAR** **MODEL**  not used in analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

def perform_differential_expression(data, normal_prefix, tumor_prefix):
    """
    Perform differential expression analysis using linear model and ANOVA.
    """
    normal_columns = [col for col in data.columns if col.startswith(normal_prefix)]
    tumor_columns = [col for col in data.columns if col.startswith(tumor_prefix)]

    results = []
    for reaction in data.index:
        normal_values = data.loc[reaction, normal_columns]
        tumor_values = data.loc[reaction, tumor_columns]

        # Create DataFrame for linear model
        df = pd.DataFrame({
            'flux': np.concatenate([normal_values, tumor_values]),
            'group': ['Healthy'] * len(normal_values) + ['Tumor'] * len(tumor_values)
        })

        # Fit linear model and perform ANOVA
        model = ols('flux ~ group', data=df).fit()
        anova = sm.stats.anova_lm(model, typ=2)


        normal_mean = np.mean(normal_values)
        tumor_mean = np.mean(tumor_values)


        if normal_mean == 0 and tumor_mean == 0:
            log_fc = 0
        elif normal_mean == 0:
            log_fc = np.log2(tumor_mean + 1)
        elif tumor_mean == 0:
            log_fc = -np.log2(normal_mean + 1)
        else:
            log_fc = np.log2(tumor_mean / normal_mean)# log fc calulation

        results.append({
            'Reaction': reaction,
            'log_fc': log_fc,
            'p_value': anova['PR(>F)'][0]
        })

    de_results = pd.DataFrame(results).dropna()
    de_results['adj_p_val'] = multipletests(de_results['p_value'], method='fdr_bh')[1]

    # threholds
    de_results['Fluxes'] = np.where(
        (de_results['log_fc'] >= 1.5) & (de_results['adj_p_val'] <= 0.01), 'Up-regulated',
        np.where(
            (de_results['log_fc'] <= -1.5) & (de_results['adj_p_val'] <= 0.01), 'Down-regulated',
            'Unchanged'
        )
    )

    return de_results

def plot_volcano(de_results, output_dir, tissue_name, full_tissue_name, arrow_scale=0.2, x_label_offset=0.3):
    """
    Plot a volcano plot for differential expression results.
    """
    plt.figure(figsize=(10, 8), dpi=300)

    for category, color in [('Up-regulated', 'red'), ('Down-regulated', 'blue'), ('Unchanged', 'gray')]:
        subset = de_results[de_results['Fluxes'] == category]
        plt.scatter(subset['log_fc'], -np.log10(subset['adj_p_val']), color=color, label=f'{category} (n={len(subset)})', alpha=0.7, s=50)

    top_up = de_results[de_results['Fluxes'] == 'Up-regulated'].nsmallest(10, 'adj_p_val')
    top_down = de_results[de_results['Fluxes'] == 'Down-regulated'].nsmallest(10, 'adj_p_val')

    x_min, x_max = plt.xlim()
    y_min, y_max = plt.ylim()

    x_padding = (x_max - x_min) * arrow_scale
    y_padding = (y_max - y_min) * arrow_scale
    plt.xlim(x_min - x_padding, x_max + x_padding)
    plt.ylim(y_min - y_padding, y_max + y_padding)

    def get_label_y_positions(data, y_min, y_max):
        return np.linspace(y_min + y_padding*0.3, y_max + y_padding*0.3, len(data))

    if not top_up.empty:
        y_positions = get_label_y_positions(top_up, y_min, y_max)
        for (_, row), y_pos in zip(top_up.iterrows(), y_positions):
            x, y = row['log_fc'], -np.log10(row['adj_p_val'])
            x_text = x_max + x_padding * x_label_offset
            plt.annotate(
                row['Reaction'],
                xy=(x, y),
                xytext=(x_text, y_pos),
                fontsize=10,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3', linewidth=0.6, color='black'),
                ha='left'
            )

    if not top_down.empty:
        y_positions = get_label_y_positions(top_down, y_min, y_max)
        for (_, row), y_pos in zip(top_down.iterrows(), y_positions):
            x, y = row['log_fc'], -np.log10(row['adj_p_val'])
            x_text = x_min - x_padding * x_label_offset
            plt.annotate(
                row['Reaction'],
                xy=(x, y),
                xytext=(x_text, y_pos),
                fontsize=10,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', linewidth=0.6, color='black'),
                ha='right'
            )


    plt.axhline(-np.log10(0.01), color='black', linestyle='dashed', linewidth=0.8)
    plt.axvline(1.5, color='black', linestyle='dashed', linewidth=0.8)
    plt.axvline(-1.5, color='black', linestyle='dashed', linewidth=0.8)

    plt.xlabel('log2 Fold Change', fontsize=14, fontweight='bold')
    plt.ylabel('-log10 FDR', fontsize=14, fontweight='bold')
    plt.title(f'{full_tissue_name} Differential Expression', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, edgecolor='black')
    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, f'{tissue_name}_volcano.png'), bbox_inches='tight', pad_inches=0.1)
    plt.close()

def plot_combined_volcano(output_dir, tissue_pairs, arrow_scale=0.15, x_label_offset=0.2):
    """
    Plot combined volcano plots
    """
    fig = plt.figure(figsize=(28, 16))
    gs = fig.add_gridspec(3, 4, height_ratios=[1.2, 1.2, 0.2])
    axes = []
    for i in range(2):
        for j in range(4):
            axes.append(fig.add_subplot(gs[i, j]))

    for i, (normal_prefix, tumor_prefix, full_tissue_name) in enumerate(tissue_pairs):
        tissue_name = normal_prefix[:-1]
        de_results = pd.read_csv(os.path.join(output_dir, f'{tissue_name}_differential_expression.csv'))

        for category, color in [('Up-regulated', 'red'), ('Down-regulated', 'blue'), ('Unchanged', 'gray')]:
            subset = de_results[de_results['Fluxes'] == category]
            axes[i].scatter(subset['log_fc'], -np.log10(subset['adj_p_val']),
                          color=color, label=category, alpha=0.7, s=50)


        axes[i].axhline(-np.log10(0.01), color='black', linestyle='dashed', linewidth=0.8)
        axes[i].axvline(1.5, color='black', linestyle='dashed', linewidth=0.8)
        axes[i].axvline(-1.5, color='black', linestyle='dashed', linewidth=0.8)

        x_min, x_max = axes[i].get_xlim()
        y_min, y_max = axes[i].get_ylim()

        x_padding = (x_max - x_min) * arrow_scale * 2.0
        y_padding = (y_max - y_min) * arrow_scale * 2.0
        axes[i].set_xlim(x_min - x_padding, x_max + x_padding)
        axes[i].set_ylim(y_min - y_padding, y_max + y_padding)

        top_up = de_results[de_results['Fluxes'] == 'Up-regulated'].nsmallest(5, 'adj_p_val')
        top_down = de_results[de_results['Fluxes'] == 'Down-regulated'].nsmallest(5, 'adj_p_val')

        def get_label_y_positions(data, y_min, y_max):
            return np.linspace(y_min + y_padding*0.3, y_max + y_padding*0.3, len(data))

        if not top_up.empty:
            y_positions = get_label_y_positions(top_up, y_min, y_max)
            for (_, row), y_pos in zip(top_up.iterrows(), y_positions):
                x, y = row['log_fc'], -np.log10(row['adj_p_val'])
                x_text = x_max + x_padding * x_label_offset
                axes[i].annotate(
                    row['Reaction'],
                    xy=(x, y),
                    xytext=(x_text, y_pos),
                    fontsize=9,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3', linewidth=0.6, color='black'),
                    ha='left'
                )

        if not top_down.empty:
            y_positions = get_label_y_positions(top_down, y_min, y_max)
            for (_, row), y_pos in zip(top_down.iterrows(), y_positions):
                x, y = row['log_fc'], -np.log10(row['adj_p_val'])
                x_text = x_min - x_padding * x_label_offset
                axes[i].annotate(
                    row['Reaction'],
                    xy=(x, y),
                    xytext=(x_text, y_pos),
                    fontsize=9,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', linewidth=0.6, color='black'),
                    ha='right'
                )

        axes[i].set_title(full_tissue_name, fontsize=16, fontweight='bold')
        axes[i].set_xlabel('log2 Fold Change', fontsize=14)
        axes[i].set_ylabel('-log10 FDR', fontsize=14)
        axes[i].tick_params(axis='both', which='major', labelsize=12)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
              loc='center',
              bbox_to_anchor=(0.5, 0.08),
              ncol=3,
              fontsize=14,
              frameon=True,
              edgecolor='black')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15, wspace=0.4, hspace=0.5)
    plt.savefig(os.path.join(output_dir, 'combined_volcano.png'), bbox_inches='tight', pad_inches=0.3, dpi=300)
    plt.close()

def main():
    """
    Main function to run the analysis.
    """
    file_path = '/content/normalized_all_samples.csv'
    output_dir = '/content/sample_data/LinearModel'
    os.makedirs(output_dir, exist_ok=True)

    data = pd.read_csv(file_path, index_col=0)
    tissue_pairs = [
        ('BLN', 'BLT', 'BronchusLung'),
        ('BN', 'BT', 'Breast'),
        ('CN', 'CT', 'Colon'),
        ('KN', 'KT', 'Kidney'),
        ('LIN', 'LIT', 'Liver'),
        ('PN', 'PT', 'Prostrate'),
        ('SN', 'ST', 'Stomach'),
        ('TN', 'TT', 'Thyroid')
    ]

    for normal_prefix, tumor_prefix, full_tissue_name in tissue_pairs:
        tissue_name = normal_prefix[:-1]
        de_results = perform_differential_expression(data, normal_prefix, tumor_prefix)
        plot_volcano(de_results, output_dir, tissue_name, full_tissue_name)
        de_results.to_csv(os.path.join(output_dir, f'{tissue_name}_differential_expression.csv'), index=False)

    plot_combined_volcano(output_dir, tissue_pairs)

if __name__ == "__main__":
    main()

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 26 and the array at index 1 has size 24

**WILCOXON**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

def perform_differential_expression(data, normal_prefix, tumor_prefix):
    """
    Perform differential expression analysis using Wilcoxon rank-sum test.

    """
    normal_columns = [col for col in data.columns if col.startswith(normal_prefix)]
    tumor_columns = [col for col in data.columns if col.startswith(tumor_prefix)]

    results = []
    for reaction in data.index:
        normal_values = data.loc[reaction, normal_columns].values.flatten()
        tumor_values = data.loc[reaction, tumor_columns].values.flatten()

        if len(normal_values) == 0 or len(tumor_values) == 0:
            continue

        normal_mean = np.mean(normal_values)
        tumor_mean = np.mean(tumor_values)


        if normal_mean == 0 and tumor_mean == 0:
            log_fc = 0  # Both are zero, assume no change
        elif normal_mean == 0:
            log_fc = np.log2(tumor_mean + 1)  # Add pseudocount when denominator is zero in either cases tumor or normal
        elif tumor_mean == 0:
            log_fc = -np.log2(normal_mean + 1)
        else:
            log_fc = np.log2(tumor_mean / normal_mean) # log fold change calculation

        statistic, p_value = stats.ranksums(normal_values, tumor_values) # ranksum from statsmodel
        results.append({'Reaction': reaction, 'log_fc': log_fc, 'p_value': p_value})

    de_results = pd.DataFrame(results).dropna()
    de_results['adj_p_val'] = multipletests(de_results['p_value'], method='fdr_bh')[1]

    # log fold change thresholds
    de_results['Fluxes'] = np.where(
        (de_results['log_fc'] >= 1.5) & (de_results['adj_p_val'] <= 0.01), 'Up-regulated',
        np.where(
            (de_results['log_fc'] <= -1.5) & (de_results['adj_p_val'] <= 0.01), 'Down-regulated',
            'Unchanged'
        )
    )

    return de_results

def plot_volcano(de_results, output_dir, tissue_name, full_tissue_name, arrow_scale=0.2, x_label_offset=0.3):
    """
    Plot a volcano plot for differential expression results.
    """
    plt.figure(figsize=(10, 8), dpi=300)

    for category, color in [('Up-regulated', 'red'), ('Down-regulated', 'blue'), ('Unchanged', 'gray')]:
        subset = de_results[de_results['Fluxes'] == category]
        plt.scatter(subset['log_fc'], -np.log10(subset['adj_p_val']), color=color, label=f'{category} (n={len(subset)})', alpha=0.7, s=50)

    top_up = de_results[de_results['Fluxes'] == 'Up-regulated'].nsmallest(10, 'adj_p_val')
    top_down = de_results[de_results['Fluxes'] == 'Down-regulated'].nsmallest(10, 'adj_p_val') # cosnidering top 10 to plot

    x_min, x_max = plt.xlim()
    y_min, y_max = plt.ylim()

    x_padding = (x_max - x_min) * arrow_scale
    y_padding = (y_max - y_min) * arrow_scale
    plt.xlim(x_min - x_padding, x_max + x_padding)
    plt.ylim(y_min - y_padding, y_max + y_padding)

    def get_label_y_positions(data, y_min, y_max):
        return np.linspace(y_min + y_padding*0.3, y_max + y_padding*0.3, len(data))

    if not top_up.empty:
        y_positions = get_label_y_positions(top_up, y_min, y_max)
        for (_, row), y_pos in zip(top_up.iterrows(), y_positions):
            x, y = row['log_fc'], -np.log10(row['adj_p_val'])
            x_text = x_max + x_padding * x_label_offset
            plt.annotate(
                row['Reaction'],
                xy=(x, y),
                xytext=(x_text, y_pos),
                fontsize=10,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3', linewidth=0.6, color='black'),
                ha='left'
            )

    if not top_down.empty:
        y_positions = get_label_y_positions(top_down, y_min, y_max)
        for (_, row), y_pos in zip(top_down.iterrows(), y_positions):
            x, y = row['log_fc'], -np.log10(row['adj_p_val'])
            x_text = x_min - x_padding * x_label_offset
            plt.annotate(
                row['Reaction'],
                xy=(x, y),
                xytext=(x_text, y_pos),
                fontsize=10,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', linewidth=0.6, color='black'),
                ha='right'
            )


    plt.axhline(-np.log10(0.01), color='black', linestyle='dashed', linewidth=0.8)
    plt.axvline(1.5, color='black', linestyle='dashed', linewidth=0.8)
    plt.axvline(-1.5, color='black', linestyle='dashed', linewidth=0.8)

    plt.xlabel('log2 Fold Change', fontsize=14, fontweight='bold')
    plt.ylabel('-log10 FDR', fontsize=14, fontweight='bold')
    plt.title(f'{full_tissue_name} Differential Expression', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, edgecolor='black')
    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, f'{tissue_name}_volcano.png'), bbox_inches='tight', pad_inches=0.1)
    plt.close()

def plot_combined_volcano(output_dir, tissue_pairs, arrow_scale=0.15, x_label_offset=0.2):
    """
    Plot combined volcano plots
    """
    fig = plt.figure(figsize=(28, 16))
    gs = fig.add_gridspec(3, 4, height_ratios=[1.2, 1.2, 0.2])
    axes = []
    for i in range(2):
        for j in range(4):
            axes.append(fig.add_subplot(gs[i, j]))

    for i, (normal_prefix, tumor_prefix, full_tissue_name) in enumerate(tissue_pairs):
        tissue_name = normal_prefix[:-1]
        de_results = pd.read_csv(os.path.join(output_dir, f'{tissue_name}_differential_expression.csv'))

        for category, color in [('Up-regulated', 'red'), ('Down-regulated', 'blue'), ('Unchanged', 'gray')]:
            subset = de_results[de_results['Fluxes'] == category]
            axes[i].scatter(subset['log_fc'], -np.log10(subset['adj_p_val']),
                          color=color, label=category, alpha=0.7, s=50)

        # Updated threshold lines to match classification
        axes[i].axhline(-np.log10(0.01), color='black', linestyle='dashed', linewidth=0.8)
        axes[i].axvline(1.5, color='black', linestyle='dashed', linewidth=0.8)
        axes[i].axvline(-1.5, color='black', linestyle='dashed', linewidth=0.8)

        x_min, x_max = axes[i].get_xlim()
        y_min, y_max = axes[i].get_ylim()

        x_padding = (x_max - x_min) * arrow_scale * 2.0
        y_padding = (y_max - y_min) * arrow_scale * 2.0
        axes[i].set_xlim(x_min - x_padding, x_max + x_padding)
        axes[i].set_ylim(y_min - y_padding, y_max + y_padding)

        top_up = de_results[de_results['Fluxes'] == 'Up-regulated'].nsmallest(5, 'adj_p_val')
        top_down = de_results[de_results['Fluxes'] == 'Down-regulated'].nsmallest(5, 'adj_p_val') # considering top five to plot in combined plot

        def get_label_y_positions(data, y_min, y_max):
            return np.linspace(y_min + y_padding*0.3, y_max + y_padding*0.3, len(data))

        if not top_up.empty:
            y_positions = get_label_y_positions(top_up, y_min, y_max)
            for (_, row), y_pos in zip(top_up.iterrows(), y_positions):
                x, y = row['log_fc'], -np.log10(row['adj_p_val'])
                x_text = x_max + x_padding * x_label_offset
                axes[i].annotate(
                    row['Reaction'],
                    xy=(x, y),
                    xytext=(x_text, y_pos),
                    fontsize=9,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3', linewidth=0.6, color='black'),
                    ha='left'
                )

        if not top_down.empty:
            y_positions = get_label_y_positions(top_down, y_min, y_max)
            for (_, row), y_pos in zip(top_down.iterrows(), y_positions):
                x, y = row['log_fc'], -np.log10(row['adj_p_val'])
                x_text = x_min - x_padding * x_label_offset
                axes[i].annotate(
                    row['Reaction'],
                    xy=(x, y),
                    xytext=(x_text, y_pos),
                    fontsize=9,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', edgecolor='gray', facecolor='white', alpha=0.8),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', linewidth=0.6, color='black'),
                    ha='right'
                )

        axes[i].set_title(full_tissue_name, fontsize=16, fontweight='bold')
        axes[i].set_xlabel('log2 Fold Change', fontsize=14)
        axes[i].set_ylabel('-log10 FDR', fontsize=14)
        axes[i].tick_params(axis='both', which='major', labelsize=12)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
              loc='center',
              bbox_to_anchor=(0.5, 0.08),
              ncol=3,
              fontsize=14,
              frameon=True,
              edgecolor='black')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15, wspace=0.4, hspace=0.5)
    plt.savefig(os.path.join(output_dir, 'combined_volcano.png'), bbox_inches='tight', pad_inches=0.3, dpi=300)
    plt.close()

def main():
    """
    Main function to run the analysis.
    """
    file_path = '/content/normalized_all_samples.csv'
    output_dir = '/content/sample_data/WILCOXON'
    os.makedirs(output_dir, exist_ok=True)

    data = pd.read_csv(file_path, index_col=0)
    tissue_pairs = [
        ('BLN', 'BLT', 'BronchusLung'),
        ('BN', 'BT', 'Breast'),
        ('CN', 'CT', 'Colon'),
        ('KN', 'KT', 'Kidney'),
        ('LIN', 'LIT', 'Liver'),
        ('PN', 'PT', 'Prostate'),
        ('SN', 'ST', 'Stomach'),
        ('TN', 'TT', 'Thyroid')
    ]

    for normal_prefix, tumor_prefix, full_tissue_name in tissue_pairs:
        tissue_name = normal_prefix[:-1]
        de_results = perform_differential_expression(data, normal_prefix, tumor_prefix)
        plot_volcano(de_results, output_dir, tissue_name, full_tissue_name)
        de_results.to_csv(os.path.join(output_dir, f'{tissue_name}_differential_expression.csv'), index=False)

    plot_combined_volcano(output_dir, tissue_pairs)

if __name__ == "__main__":
    main()

ZeroDivisionError: float division by zero